## Data windowing

Time series forecasting with deep learning methods relies on creating appropriate time windows (inputs) of consecutive samples from the data and specifying the labels.

The main features of the input windows are:

- The width (number of time steps) of the input and label windows.
- The time offset between them.
- Which features are used as inputs, labels, or both.

![Windowing features](../references/time_series_windowing.png)

Depending on the task and type of model we may want to generate a variety of data windows for:

1. Single-Step Models:

    - Definition: These models predict the value of the time series at the next time step, given the current and previous values.    - Example: Given data at time  $t−2$, $t−1$, $t$ predict the value at time $t+1$.
    - Use Case: When the prediction requirement is only for the next immediate time step.
    - Python Libraries: statsmodels, scikit-learn, Keras/TensorFlow.

2. Multi-Step Models:

    - Definition: These models predict the values of the time series for multiple future time steps, not just the next one.
    - Example: Given data at time  $t−2$, $t−1$, $t$ predict the value at time $t+1$, $t+2$, ..., $t+n$.
    - Use Case: When predictions for a longer horizon are needed.
    - Python Libraries: Prophet, GluonTS, Keras/TensorFlow.

3. Multi-Output Models:

    Definition: These models predict multiple values at each time step. Typically used when predicting multiple related time series simultaneously.
    Example: Predicting temperature, humidity, and wind speed at the same future time steps.
    Use Case: Useful when forecasting multiple related variables.
    Python Libraries: scikit-learn, Keras/TensorFlow with multi-output functionality.

4. Single-Shot Models:

    Definition: These models generate predictions for multiple future time steps in a single operation, rather than predicting one step at a time.
    Example: A model that directly predicts $t+1$, $t+2$, ..., $t+n$ in one go, rather than looping through each prediction step-by-step.
    Use Case: When the model is designed to handle multiple time steps together for computational efficiency.
    Python Libraries: Keras/TensorFlow, especially with sequence-to-sequence architectures.

    

In [1]:
import numpy as np
import tensorflow as tf

In [2]:
X = np.arange(100)
Y = X*2

print(X)
print(Y)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]
[  0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34
  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70
  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100 102 104 106
 108 110 112 114 116 118 120 122 124 126 128 130 132 134 136 138 140 142
 144 146 148 150 152 154 156 158 160 162 164 166 168 170 172 174 176 178
 180 182 184 186 188 190 192 194 196 198]


In [9]:
sample_length = 15
input_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X,
    targets=None,
    sequence_length=sample_length,
    sequence_stride=sample_length,
)

In [10]:
for batch in input_dataset:
    display(batch)

<tf.Tensor: shape=(6, 15), dtype=int32, numpy=
array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
       [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
       [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44],
       [45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
       [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74],
       [75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]])>

In [13]:
target_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=Y,
    targets=None,
    sequence_length=sample_length,
    sequence_stride=1
)

In [14]:
for batch in target_dataset:
    display(batch)

<tf.Tensor: shape=(86, 15), dtype=int32, numpy=
array([[  0,   2,   4, ...,  24,  26,  28],
       [  2,   4,   6, ...,  26,  28,  30],
       [  4,   6,   8, ...,  28,  30,  32],
       ...,
       [166, 168, 170, ..., 190, 192, 194],
       [168, 170, 172, ..., 192, 194, 196],
       [170, 172, 174, ..., 194, 196, 198]])>

### The `WindowGenerator` class

We define a `WindowGenerator` class which can:

1. Handle the indexes and offsets.
2. Split windows of features into (features, labels) pairs.
3. Plot the content of the resulting windows.
4. Efficiently generate batches of these windows from the training, evaluation, and test data, using `tf.data.Datasets`.

In [ ]:
# open_file = open(file_name, "rb")
# loaded_list = pickle.load(open_file)
# open_file.close()
# print(loaded_list)